# Initialize, sample, and review SpecMine data

Run the cells from top to bottom. `CURATED_ONLY = True` selects the curated repositories; set it to `False` to use the full metadata table. Metadata and feature parquets are read from `notebooks/data` when cached and downloaded from Hugging Face when missing. Curated metadata is cached after its first load, and only missing Markdown files for the requested random sample are downloaded. The notebook then chooses one unreviewed repository from that sample and displays its Markdown, precomputed features, and metadata. The first run and any newly selected uncached spec may require network access; rerunning the same seeded sample reuses its local files. Set `SEED = None` for a new random selection each run, or choose an integer seed to reproduce a sample.

In [ ]:
import html
import sys
from pathlib import Path

from IPython.display import HTML, display

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src" / "specmine").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/specmine. Open this notebook from the project "
        "folder or one of its subfolders."
    )
sys.path.insert(0, str(project_root / "src"))

from specmine.data import (
    download_random_spec_sample,
    load_data,
    load_feature_data,
)
from specmine.labeling import match_review_features, select_review_spec
from specmine.labels import read_labels_csv

SEED = None
SPEC_SAMPLE_SIZE = 3
CURATED_ONLY = True
DATA_DIR = project_root / "notebooks" / "data"
REAL_LABELS_CSV = project_root / "examples" / "real_labels.csv"
FAKE_LABELS_CSV = project_root / "examples" / "fake_labels.csv"

print(f"Data directory: {DATA_DIR}")
print(f"Requested distinct spec/repository count: {SPEC_SAMPLE_SIZE}")

In [ ]:
if SPEC_SAMPLE_SIZE < 1:
    raise ValueError("SPEC_SAMPLE_SIZE must be a positive integer.")

metadata = load_data(
    curated_only=CURATED_ONLY,
    data_dir=DATA_DIR,
    download_raw_specs=False,
)
if metadata is None or metadata.empty:
    raise RuntimeError("The SpecMine metadata table is empty or unavailable.")

sampled_specs = download_random_spec_sample(
    metadata,
    SPEC_SAMPLE_SIZE,
    seed=SEED,
    data_dir=DATA_DIR,
)
if sampled_specs.empty:
    raise RuntimeError(
        "No raw spec files were available for the requested sample."
    )
if len(sampled_specs) > SPEC_SAMPLE_SIZE:
    raise RuntimeError("The sample exceeds SPEC_SAMPLE_SIZE.")
if sampled_specs["file_url_sha16"].nunique() != len(sampled_specs):
    raise RuntimeError("The sample contains duplicate spec IDs.")
if sampled_specs["repo_id"].nunique() != len(sampled_specs):
    raise RuntimeError("The sample contains duplicate repository IDs.")

manifest_path = DATA_DIR / "spec_sample.csv"
sampled_specs.to_csv(manifest_path, index=False)
print(
    f"Ready: {len(sampled_specs)} of up to {SPEC_SAMPLE_SIZE} requested "
    f"unique spec files from "
    f"{sampled_specs['repo_id'].nunique()} unique repositories."
)
print(f"Sample manifest: {manifest_path}")

## Choose one repository to review

The next cell chooses one random repository from the downloaded sample. It checks the real and fake label files for repository overlap and excludes repositories that already appear in either file. It selects one spec from the chosen repository and keeps its IDs in `review_spec` for the display cell below.

In [ ]:
real_labels = read_labels_csv(REAL_LABELS_CSV)
fake_labels = read_labels_csv(FAKE_LABELS_CSV)
review_spec = select_review_spec(
    sampled_specs,
    real_labels,
    fake_labels,
    seed=SEED,
    label_metadata=metadata,
)
if len(review_spec) != 1:
    raise RuntimeError("Repository selection must contain exactly one spec.")

review_row = review_spec.iloc[0]
print(f"Selected repository: {review_row['repo_name']}")
print(f"repo_id: {review_row['repo_id']}")
print(f"spec_id: {review_row['file_url_sha16']}")
print(f"Spec file: {review_row['file_path']}")

## Spec file, features, and metadata

This cell uses the repository/spec chosen above; no IDs need to be copied manually. It displays the selected file's exact precomputed feature row and useful repository/file attributes. Commit history is intentionally not displayed or used.

In [ ]:
feature_rows = load_feature_data(
    review_spec[["file_url_sha16"]],
    data_dir=DATA_DIR,
)
review_features = match_review_features(review_spec, feature_rows)

display_columns = [
    "repo_id",
    "repo_name",
    "repo_description",
    "repo_language",
    "repo_stars",
    "repo_topics",
    "file_url_sha16",
    "file_path",
    "file_name",
    "file_sha",
    "file_content_sha",
    "file_content_bytes",
    "content_fetch_status",
    "spec_tool",
    "spec_role",
]
available_columns = [column for column in display_columns if column in review_spec]
display(HTML("<h3>Repository and spec attributes</h3>"))
display(review_spec[available_columns].reset_index(drop=True))
display(HTML("<h3>Precomputed content features</h3>"))
display(review_features)

spec_path = Path(review_row.local_path)
if not spec_path.is_file():
    raise FileNotFoundError(f"Selected spec file is missing: {spec_path}")
spec_markdown = spec_path.read_text(encoding="utf-8")
display(HTML("<h3>Spec Markdown</h3>"))
display(HTML("<pre>" + html.escape(spec_markdown) + "</pre>"))

print(
    "To record a manual review, add repo_id,spec_id,label,confidence to "
    "examples/real_labels.csv. Use label 0, 1, or blank if ambiguous; "
    "confidence is 0-100 for a nonblank label."
)